In [4]:
"""
build_parquet.py
================
Converts the LOTO-RO knowledge graph from a structured Excel workbook
(four sheets: entities, relationships, text_units, attributes) into the
four Parquet files consumed by the GraphRAG pipeline.

Only the fields actively used by the pipeline are written to each file.
See pipeline documentation for the complete field-to-module mapping.

Input
-----
- Grafo_nuovo.xlsx  : Excel workbook with sheets:
                      'entità', 'relazioni', 'testi', 'attributes'

Output (data/input/)
-----
- entities.parquet
- relationships.parquet
- text_units.parquet
- attributes.parquet

Usage
-----
    python build_parquet.py
"""

import ast
import os
import re
import datetime

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

FILE = "./LOTO-RO_grounded_KG.xlsx"
OUT  = "graphrag/data/input"

os.makedirs(OUT, exist_ok=True)

sheets  = pd.read_excel(FILE, sheet_name=None)
ent     = sheets["entità"]
rel     = sheets["relazioni"]
txt     = sheets["testi"]
cov_raw = sheets["attributes"]

# PyArrow type aliases
STR = pa.string()
F64 = pa.float64()
I64 = pa.int64()
LST = pa.list_(pa.string())

# ── Helpers ───────────────────────────────────────────────────────────────────

_ISO_DATE_RE = re.compile(r"\b(\d{4})-(\d{2})-(\d{2})\b")


def normalize_dates_in_text(s: str) -> str:
    """Converts ISO dates (YYYY-MM-DD) to DD/MM/YYYY inside free text."""
    if not s:
        return s
    return _ISO_DATE_RE.sub(
        lambda m: f"{m.group(3)}/{m.group(2)}/{m.group(1)}", s
    )


def to_list(s) -> list:
    """
    Converts any cell value to a plain Python list of strings.
    Handles: Python list literals, comma-separated strings, NaN, empty.
    """
    if pd.isna(s) or str(s).strip() in ("", "nan", "[]"):
        return []
    s = str(s).strip()
    if s.startswith("["):
        try:
            return [str(x).strip() for x in ast.literal_eval(s) if str(x).strip()]
        except (ValueError, SyntaxError):
            pass
    return [x.strip() for x in s.split(",") if x.strip()]


def safe(v) -> str:
    """Converts any value to string, replacing NaN with empty string."""
    s = str(v).strip()
    return "" if s == "nan" else s


def safe_text(v) -> str:
    """Converts to string and normalises ISO dates to DD/MM/YYYY."""
    return normalize_dates_in_text(safe(v))


def save(df: pd.DataFrame, path: str, schema: pa.Schema) -> None:
    """Writes a DataFrame to Parquet with the specified Arrow schema."""
    table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
    pq.write_table(table, path)
    print(f"  {path}  ({len(df)} rows)")


# ── ENTITIES ──────────────────────────────────────────────────────────────────
# Fields used by the pipeline:
#   id            → node identifier; type/title/attribute indices
#   title         → title index; parser prompt; subgraph serialisation
#   type          → type index; visualisation colouring
#   description   → subgraph serialisation to LLM context
#   text_unit_ids → provenance chain: node → source narratives
#   frequency     → pre-computed count for LLM quantitative claims
#   degree        → top-N trimming in visualisation

ent_out = pd.DataFrame({
    "id":    ent["id"].apply(safe),
    "title": ent["title"].apply(safe),
    "type":  ent["type"].apply(safe),
    "description":   ent["description"].apply(safe_text),
    "text_unit_ids": ent["text_unit"].apply(to_list),
    "frequency": (
        pd.to_numeric(ent["frequency"], errors="coerce")
        .fillna(0).astype("int64")
    ),
    "degree": (
        pd.to_numeric(ent["degree"], errors="coerce")
        .fillna(0).astype("int64")
    ),
})

save(
    ent_out,
    f"{OUT}/entities.parquet",
    pa.schema([
        pa.field("id",            STR),
        pa.field("title",         STR),
        pa.field("type",          STR),
        pa.field("description",   STR),
        pa.field("text_unit_ids", LST),
        pa.field("frequency",     I64),
        pa.field("degree",        I64),
    ])
)

# Build degree lookup for combined_degree computation in relationships
degree_map = dict(zip(ent_out["id"], ent_out["degree"]))


# ── RELATIONSHIPS ─────────────────────────────────────────────────────────────
# Fields used by the pipeline:
#   id              → edge attribute; subgraph serialisation
#   Source_id       → edge source endpoint (graph assembly)
#   Target_id       → edge target endpoint (graph assembly)
#   description     → subgraph serialisation to LLM context
#   weight          → edge width in visualisation
#   combined_degree → edge attribute
#   text_unit_ids   → provenance chain: edge → source narratives

rel_out = pd.DataFrame({
    "id":        rel["id"].apply(safe),
    "Source_id": rel["source_id"].apply(safe),
    "Target_id": rel["target_id"].apply(safe),
    "description":   rel["description"].apply(safe_text),
    "weight": (
        pd.to_numeric(rel["weight"], errors="coerce")
        .fillna(1.0).astype("float64")
    ),
    "text_unit_ids": rel["text_unit"].apply(to_list),
    "combined_degree": rel.apply(
        lambda r:
            degree_map.get(safe(r["source_id"]), 0) +
            degree_map.get(safe(r["target_id"]), 0),
        axis=1
    ).astype("int64"),
})

save(
    rel_out,
    f"{OUT}/relationships.parquet",
    pa.schema([
        pa.field("id",              STR),
        pa.field("Source_id",       STR),
        pa.field("Target_id",       STR),
        pa.field("description",     STR),
        pa.field("weight",          F64),
        pa.field("combined_degree", I64),
        pa.field("text_unit_ids",   LST),
    ])
)


# ── TEXT UNITS ────────────────────────────────────────────────────────────────
# Fields used by the pipeline:
#   id   → re-indexed at SearchEngine init for O(1) lookup
#   text → source narrative chunk passed to LLM as context

txt_out = pd.DataFrame({
    "id":   txt["id"].apply(safe),
    "text": txt["text"].apply(safe_text),
})

save(
    txt_out,
    f"{OUT}/text_units.parquet",
    pa.schema([
        pa.field("id",   STR),
        pa.field("text", STR),
    ])
)


# ── attributeS ────────────────────────────────────────────────────────────────
# Fields used by the pipeline:
#   subject_id        → joined to entities.title to attach attributes to nodes
#   type              → inline attribute label in subgraph serialisation
#   description       → inline attribute value in subgraph serialisation

cov_out = pd.DataFrame({
    "id":          cov_raw["id"].apply(safe),
    "subject_id":  cov_raw["subject_id"].apply(safe),
    "type":        cov_raw["type"].apply(safe),
    "description": cov_raw["description"].apply(safe_text),
})

save(
    cov_out,
    f"{OUT}/attributes.parquet",
    pa.schema([
        pa.field("id",          STR),
        pa.field("subject_id",  STR),
        pa.field("type",        STR),
        pa.field("description", STR),
    ])
)

print("\nAll Parquet files written to", OUT)

  graphrag/data/input/entities.parquet  (137 rows)
  graphrag/data/input/relationships.parquet  (525 rows)
  graphrag/data/input/text_units.parquet  (20 rows)
  graphrag/data/input/attributes.parquet  (132 rows)

All Parquet files written to graphrag/data/input
